# Task 3: Qwen2.5-Coder-7B QLoRA SFT & Debugging Pipeline

Pipeline for Vietnamese Pedagogical Feedback Generation (Task 3).
- **Architecture**: Qwen2.5-Coder-7B-Instruct with 4-bit NormalFloat (NF4) double quantization.
- **Data Strategy**: Mixed balanced fine-tuning (real student submissions + verified synthetic pairs).
- **Execution Mode**: `DEBUG_MODE=True` for rapid 15-step smoke test and real-val slice validation; `DEBUG_MODE=False` for production training and 328-sample public inference.


In [ ]:
# ==============================================================================
# Cell 1: Pipeline Configuration & Execution Flags
# ==============================================================================
import os

# Repository & VCS configuration
REPO_URL = "github.com/tiendinhquang2104/llm-final-project.git"
BRANCH = "fix/task3-strict-feedback"
WORK_DIR = "/kaggle/tmp"

# Execution mode: True = Smoke test 15 steps + real-val eval; False = Full training + public eval
DEBUG_MODE: bool = True
SMOKE_STEPS: int = 15

# Experiment settings
EXPERIMENT: str = "mixed_balanced"
EPOCHS: int = 1
MAX_STEPS: int = SMOKE_STEPS if DEBUG_MODE else None
EVAL_SPLIT: str = "real-val" if DEBUG_MODE else "public"

# Dataset sources (fallback Google Drive IDs)
DRIVE_ID_DATA = "1gfMQ9LpCbg2AOwPfHnt7qe220J1AfLBj"
DRIVE_ID_SYNTH = "1oESzEa-vwppDD-dDTmAVk7KIHqUsgowl"

print(f"[CONFIG] Mode: {'DEBUG' if DEBUG_MODE else 'PRODUCTION'} | Split: {EVAL_SPLIT} | Steps: {MAX_STEPS}")


In [ ]:
# ==============================================================================
# Cell 2: Workspace Initialization & Environment Setup
# ==============================================================================
import subprocess, sys, os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

def resolve_secret(key: str) -> str:
    try:
        return UserSecretsClient().get_secret(key) or ""
    except Exception:
        return ""

token = resolve_secret("GITHUB_TOKEN")
clone_url = f"https://{token}@{REPO_URL}" if token else f"https://{REPO_URL}"

repo_path = Path(WORK_DIR) / "repo"
if not repo_path.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, clone_url, str(repo_path)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_path), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo_path), "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

# Propagate credentials
env_vars = {}
for secret_key in ["GEMINI_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"]:
    val = resolve_secret(secret_key)
    if val:
        os.environ[secret_key] = val
        env_vars[secret_key] = val

if env_vars:
    env_content = "".join(f"{k}={v}\n" for k, v in env_vars.items())
    Path(".env").write_text(env_content, encoding="utf-8")

head_commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print(f"[ENV] Workspace: {os.getcwd()} | Current commit: {head_commit}")


In [ ]:
# ==============================================================================
# Cell 3: Dependencies Installation & CUDA Hardware Sanity
# ==============================================================================
!pip install -q gdown pyyaml pytest
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes peft trl datasets

import torch
assert torch.cuda.is_available(), "CUDA runtime is required."
device_name = torch.cuda.get_device_name(0)
total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"[HARDWARE] GPU 0: {device_name} | VRAM: {total_vram:.2f} GiB | Visible GPUs: {torch.cuda.device_count()}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader


In [ ]:
# ==============================================================================
# Cell 4: Data Ingestion & Preprocessing
# ==============================================================================
import os, shutil, glob, gdown, subprocess, zipfile
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/synth").mkdir(parents=True, exist_ok=True)
download_dir = Path(WORK_DIR) / "download"
download_dir.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(zip_path: Path, target_dir: Path):
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)

# 1. Resolve raw datasets (sample_dataset and public_test_release)
sample_found = glob.glob("/kaggle/input/**/sample_dataset", recursive=True)
public_found = glob.glob("/kaggle/input/**/public_test_release", recursive=True)

if sample_found and not Path("data/raw/sample_dataset").exists():
    shutil.copytree(sample_found[0], "data/raw/sample_dataset")
if public_found and not Path("data/raw/public_test_release").exists():
    shutil.copytree(public_found[0], "data/raw/public_test_release")

if not (Path("data/raw/sample_dataset/exams.json").exists() and Path("data/raw/public_test_release/exams.json").exists()):
    zip_dest = download_dir / "data.zip"
    if not zip_dest.exists() and DRIVE_ID_DATA:
        gdown.download(id=DRIVE_ID_DATA, output=str(zip_dest), quiet=False)
    
    if zip_dest.exists():
        safe_extract_zip(zip_dest, Path("data/raw"))
    # Unpack any bundle or nested zip archives
    for z in list(Path("data/raw").glob("*.zip")) + list(download_dir.glob("*.zip")):
        try:
            safe_extract_zip(z, Path("data/raw"))
        except Exception:
            pass

# Flatten any nested split directories
for split_name in ["sample_dataset", "public_test_release"]:
    nested = Path(f"data/raw/{split_name}/{split_name}")
    if nested.exists() and nested.is_dir():
        for item in nested.iterdir():
            shutil.move(str(item), str(nested.parent))
        nested.rmdir()

shutil.rmtree("data/raw/__MACOSX", ignore_errors=True)

# 2. Resolve synthetic dataset
synth_target = Path("data/synth/generate-dataset")
synth_found = [d for d in glob.glob("/kaggle/input/**/generate-dataset*", recursive=True) if os.path.isdir(d) and (Path(d) / "exams.json").exists()]

if synth_found and not synth_target.exists():
    shutil.copytree(synth_found[0], synth_target)
elif not synth_target.exists() and DRIVE_ID_SYNTH:
    synth_zip = download_dir / "synth.zip"
    if not synth_zip.exists():
        gdown.download(id=DRIVE_ID_SYNTH, output=str(synth_zip), quiet=False)
    if synth_zip.exists():
        safe_extract_zip(synth_zip, Path("data/synth"))
        nested_synth = Path("data/synth/generate-dataset/generate-dataset")
        if nested_synth.exists() and nested_synth.is_dir():
            for item in nested_synth.iterdir():
                shutil.move(str(item), str(nested_synth.parent))
            nested_synth.rmdir()

shutil.rmtree("data/synth/__MACOSX", ignore_errors=True)

# 3. Ensure public requested levels mapping exists
req_levels_path = Path("data/raw/public_test_release/task3_requested_levels.json")
if not req_levels_path.exists() and Path("data/raw/public_test_release/task3_feedback.json").exists():
    with open("data/raw/public_test_release/task3_feedback.json", "r", encoding="utf-8-sig") as f:
        public_fb = json.load(f)
    req_map = {
        s["sample_id"]: s["input"]["target_feedback_level"]
        for s in public_fb.get("samples", [])
        if "sample_id" in s and "input" in s and "target_feedback_level" in s["input"]
    }
    with open(req_levels_path, "w", encoding="utf-8") as f:
        json.dump(req_map, f, indent=2, ensure_ascii=False)
    print(f"[DATA] Created public requested levels: {len(req_map)} entries.")

# 4. Verify and execute Task 3 dataset preparation
assert Path("data/raw/sample_dataset/exams.json").exists(), "Missing data/raw/sample_dataset/exams.json"
assert Path("data/raw/public_test_release/exams.json").exists(), "Missing data/raw/public_test_release/exams.json"
assert req_levels_path.exists(), "Missing task3_requested_levels.json"

subprocess.run(["python", "scripts/task3_prepare.py", "--filter-synthetic", "strict"], check=True)
print("[DATA] Partitions prepared successfully.")


In [ ]:
# ==============================================================================
# Cell 5: QLoRA Training / Smoke Debugging Engine
# ==============================================================================
import yaml, shutil, subprocess, os
from pathlib import Path

adapter_dir = f"checkpoints/task3/{EXPERIMENT}"
working_adapter = "/kaggle/working/task3_lora_adapter"
checkpoint_exists = any(
    Path(p).exists() for p in [
        f"{working_adapter}/adapter_model.safetensors",
        f"{adapter_dir}/adapter_model.safetensors",
        f"{adapter_dir}/adapter/adapter_model.safetensors"
    ]
)

if checkpoint_exists and not DEBUG_MODE:
    print("[TRAIN] Existing adapter detected. Skipping training stage.")
else:
    cfg_path = Path("configs/task3/qlora.yaml")
    if not cfg_path.exists():
        cfg_path.write_text(yaml.safe_dump({
            "extends": "baseline.yaml",
            "experiment": EXPERIMENT,
            "output_dir": adapter_dir,
            "min_free_vram_gib": 6,
            "attn_implementation": "sdpa",
            "max_seq_length": 4096,
            "lora_r": 16,
            "lora_alpha": 32,
            "lora_dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            "optim": "paged_adamw_8bit",
            "learning_rate": 0.00002,
            "lr_scheduler_type": "cosine",
            "warmup_ratio": 0.03,
            "max_grad_norm": 1.0,
            "epochs": EPOCHS,
            "gradient_accumulation_steps": 4,
            "ignore_competing_gpu_workload": True,
            "device_map": "auto"
        }), encoding="utf-8")

    train_cmd = [
        "python", "scripts/train_task3.py",
        "--config", str(cfg_path),
        "--experiment", EXPERIMENT,
        "--output-dir", adapter_dir
    ]
    if MAX_STEPS:
        train_cmd.extend(["--max-steps", str(MAX_STEPS)])

    print("[TRAIN] Executing:", " ".join(train_cmd))
    train_env = os.environ.copy()
    train_env["CUDA_VISIBLE_DEVICES"] = "0"
    train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    subprocess.run(train_cmd, env=train_env, check=True)

    source_dir = f"{adapter_dir}/adapter" if Path(f"{adapter_dir}/adapter").exists() else adapter_dir
    shutil.copytree(source_dir, working_adapter, dirs_exist_ok=True)
    print(f"[TRAIN] Adapter synchronized to {working_adapter}")


In [ ]:
# ==============================================================================
# Cell 6: Bounded Batch Inference
# ==============================================================================
import os, subprocess, glob
from pathlib import Path

mode_str = "debug" if DEBUG_MODE else "prod"
output_file = f"outputs/task3/{mode_str}_{EVAL_SPLIT}.predictions.json"
Path("outputs/task3").mkdir(parents=True, exist_ok=True)

taxonomy_file = "outputs/task3/public_baseline_taxonomy.json"
if EVAL_SPLIT == "public":
    from scripts.build_task3_submission import build_default_taxonomy
    if not Path(taxonomy_file).exists():
        build_default_taxonomy(Path("data/raw/public_test_release"), Path(taxonomy_file))

infer_cmd = [
    "python", "scripts/run_task3.py",
    "--split", EVAL_SPLIT,
    "--output", output_file,
    "--context-limit", "8192",
    "--resume-progress"
]

if EVAL_SPLIT == "public":
    req_file = "data/raw/public_test_release/task3_requested_levels.json"
    infer_cmd.extend([
        "--taxonomy-source", "task2-json",
        "--taxonomy-json", taxonomy_file,
        "--allow-unverified-task2-export",
        "--requested-levels", req_file
    ])
else:
    infer_cmd.extend(["--taxonomy-source", "gold"])

target_adapter = working_adapter
input_adapters = glob.glob("/kaggle/input/**/task3_lora_adapter", recursive=True)
if Path(f"{working_adapter}/adapter_model.safetensors").exists() or Path(f"{working_adapter}/adapter_model.bin").exists():
    target_adapter = working_adapter
elif input_adapters and (Path(f"{input_adapters[0]}/adapter_model.safetensors").exists() or Path(f"{input_adapters[0]}/adapter_model.bin").exists()):
    target_adapter = input_adapters[0]
elif Path(f"{adapter_dir}/adapter").exists():
    target_adapter = f"{adapter_dir}/adapter"
else:
    target_adapter = adapter_dir

infer_cmd.extend(["--config", "configs/task3/baseline.yaml", "--adapter", str(target_adapter)])

infer_env = os.environ.copy()
infer_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if "CUDA_VISIBLE_DEVICES" in infer_env:
    del infer_env["CUDA_VISIBLE_DEVICES"]

print(f"[INFER] Split: {EVAL_SPLIT} | Output: {output_file}")
print("[INFER] Executing:", " ".join(infer_cmd))
subprocess.run(infer_cmd, env=infer_env, check=True)


In [ ]:
# ==============================================================================
# Cell 7: Compliance Evaluation & Submission Packaging
# ==============================================================================
import json, shutil, subprocess
from pathlib import Path

traces_file = output_file.replace(".predictions.json", ".predictions.traces.json")
eval_file = output_file.replace(".predictions.json", ".evaluation.json")

eval_cmd = [
    "python", "scripts/evaluate_task3.py",
    "--data-dir", "data/processed/task3",
    "--split", EVAL_SPLIT,
    "--predictions", output_file,
    "--traces", traces_file,
    "--taxonomy-source", "task2-json" if EVAL_SPLIT == "public" else "gold",
    "--output", eval_file
]
subprocess.run(eval_cmd, check=True)

if Path(eval_file).exists():
    report = json.loads(Path(eval_file).read_text(encoding="utf-8"))
    print("[EVAL METRICS]")
    print(json.dumps(report.get("summary", report), indent=2, ensure_ascii=False))

if EVAL_SPLIT == "public":
    submission_path = "/kaggle/working/task3_submission.json"
    shutil.copy(output_file, submission_path)
    print(f"[EXPORT] Final submission written to {submission_path}")
